# 03 — Da detecção à decisão: política A1 em dry-run

**Grupo 1 · Tema G2 — Anomalia de carga**

O card do G2 pede: *quando a maioria da janela for anômala, gerar uma decisão e uma
política candidata do tipo "reduzir congestionamento UL" em dry-run*.

Este notebook fecha o fio condutor da disciplina — **fonte → indicador → análise →
decisão** — reproduzindo a mecânica de janela e voto do lab e gerando a política do
grupo.

> **Dry-run.** Tudo aqui é execução simulada. Nenhuma requisição é enviada ao Near-RT
> RIC e **nenhum efeito físico na RAN é alegado**.

In [1]:
import sys
from pathlib import Path

sys.path.append("..")          # code/ -> g2_lib.py
from g2_lib import *           # noqa: F403

RAIZ = Path("../..").resolve()
AMOSTRA = RAIZ / "code" / "datasets" / "kpm-ue-tp-sample"
DERIVED = RAIZ / "derived"
FIGURES = RAIZ / "figures"
DERIVED.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

print("raiz do pacote:", RAIZ.name)
print("fonte:", (AMOSTRA / "kpm.sqlite").relative_to(RAIZ))

raiz do pacote: _pr
fonte: code\datasets\kpm-ue-tp-sample\kpm.sqlite


In [2]:
import pandas as pd

amostras = carregar_amostras(AMOSTRA / "kpm.sqlite")
modelo = treinar(amostras[amostras.phase == "baseline"], tratar_delay_zero=True)
escorado = escorar_df(amostras, modelo)
print("modelo V2 · limiar", modelo["score_threshold"],
      "· mínimo", modelo["min_anomalous_features"], "features · janela", JANELA)

modelo V2 · limiar 3.5 · mínimo 2 features · janela 5


## 1. A regra de decisão

```
janela  = últimas 5 amostras
votos   = nº de amostras com ≥ 2 features anômalas
decisão = "apply" se votos > 2,5 ; senão "observe"
```

A janela existe justamente para responder à pergunta do tema: **sustentado, não pico
isolado**. Uma amostra anômala solitária nunca move a decisão.

In [3]:
por_fase = []
for fase in ORDEM_FASES:
    bloco = escorado[escorado.phase == fase]
    aval = decidir_janela(bloco, JANELA)
    por_fase.append({
        "fase": fase, "votos": f"{aval['apply_votes']}/{aval['window_size']}",
        "decisao": aval["decision"], "ISC_ultima": aval["latest"]["isc"],
        "features_anomalas": ", ".join(aval["latest"]["anomalous_features"]) or "—",
    })
display(pd.DataFrame(por_fase))

,fase,votos,decisao,ISC_ultima,features_anomalas
0,baseline,0/5,observe,0.00,—
1,stress,5/5,apply,6.69,"DRB.UEThpUl, RRU.PrbTotUl"
2,recovery,0/5,observe,0.00,—


> A decisão é `apply` **apenas** ao final da fase de stress. As janelas finais de
> baseline e de recovery votam `observe` — inclusive a de recovery, apesar de a fase
> conter uma amostra anômala isolada. É a janela fazendo o trabalho que se espera dela.

## 2. Varredura da regra ao longo do experimento

Onde a política seria acionada se o detector rodasse continuamente, amostra a amostra?

In [4]:
seq = escorado.reset_index(drop=True)
votos = seq["anomala"].rolling(JANELA, min_periods=JANELA).sum()
seq["votos_janela"] = votos
seq["decisao_janela"] = (votos > JANELA / 2).map({True: "apply", False: "observe"})

resumo = (seq.dropna(subset=["votos_janela"])
          .groupby(["phase", "decisao_janela"], observed=True).size()
          .unstack(fill_value=0))
display(resumo)

acionamentos = seq[(seq["decisao_janela"] == "apply") &
                   (seq["decisao_janela"].shift() != "apply")]
print(f"a politica seria acionada {len(acionamentos)} vez(es) em 100 amostras:")
display(acionamentos[["phase", "sample_index", "votos_janela", "isc"]])

decisao_janela,apply,observe
phase,,
baseline,0,16
stress,57,3
recovery,3,17


a politica seria acionada 1 vez(es) em 100 amostras:


,phase,sample_index,votos_janela,isc
23,stress,3,3.0,6.70033


> Um único acionamento, no início da fase de stress, e nenhum nas fases calmas. Em um
> cenário de operação isso é o comportamento desejado: a política dispara uma vez
> quando o regime muda, e não a cada oscilação.

## 3. A decisão e a política do grupo

Geramos `derived/decision_g1.json` no mesmo formato do `decision.json` do lab, para que
seja comparável com o artefato do docente.

In [5]:
stress = escorado[escorado.phase == "stress"]
avaliacao = decidir_janela(stress, JANELA)
politica = construir_politica(avaliacao, modelo)

decisao_g1 = {
    "grupo": "01", "tema": "G2 — Anomalia de carga",
    "run_id": str(amostras["run_id"].iloc[0]),
    "modelo": {k: modelo[k] for k in
               ["algorithm", "score_threshold", "min_anomalous_features",
                "mad_floor", "tratar_delay_zero_como_nulo"]},
    "evaluation": avaliacao,
    "policy": politica,
    "actuation": {"mode": "dry-run", "real": {}},
}
salvar_json(decisao_g1, DERIVED / "decision_g1.json")
print("gravado:", (DERIVED / "decision_g1.json").relative_to(RAIZ), "\n")
import json; print(json.dumps(decisao_g1, indent=2, ensure_ascii=False)[:1500], "...")

gravado: derived\decision_g1.json 

{
  "grupo": "01",
  "tema": "G2 — Anomalia de carga",
  "run_id": "ue-tp-20260804-174422",
  "modelo": {
    "algorithm": "robust-baseline-mad",
    "score_threshold": 3.5,
    "min_anomalous_features": 2,
    "mad_floor": 1.0,
    "tratar_delay_zero_como_nulo": true
  },
  "evaluation": {
    "evaluated_at": "2026-08-29T13:31:44.209883+00:00",
    "window_size": 5,
    "apply_votes": 5,
    "decision": "apply",
    "latest": {
      "phase": "stress",
      "sample_index": 59,
      "sample": {
        "DRB.RlcSduDelayDl": 154.88,
        "DRB.UEThpUl": 77216.56,
        "RRU.PrbTotUl": 99.0
      },
      "scores": {
        "DRB.RlcSduDelayDl": 0.26,
        "DRB.UEThpUl": 77212.84,
        "RRU.PrbTotUl": 97.0
      },
      "isc": 6.69,
      "anomalous_features": [
        "DRB.UEThpUl",
        "RRU.PrbTotUl"
      ]
    }
  },
  "policy": {
    "ric_id": "ric-oran",
    "policy_id": "g2-load-anomaly-20260829133144",
    "service_id": "grupo-

## 4. Comparação com a decisão do docente

Nosso pipeline chega à mesma decisão pelo mesmo caminho — evidência de que a
divergência V1/V2 do notebook 02 é deliberada, não acidental.

In [6]:
oficial = json.loads((AMOSTRA / "decision.json").read_text(encoding="utf-8"))["evaluation"]
display(pd.DataFrame([
    {"origem": "docente (decision.json)", "decisao": oficial["decision"],
     "votos": f"{oficial['apply_votes']}/{oficial['window_size']}",
     "features_anomalas": len(oficial["latest"]["anomalous_features"])},
    {"origem": "grupo 01 (decision_g1.json)", "decisao": avaliacao["decision"],
     "votos": f"{avaliacao['apply_votes']}/{avaliacao['window_size']}",
     "features_anomalas": len(avaliacao["latest"]["anomalous_features"])},
]))

,origem,decisao,votos,features_anomalas
0,docente (decision.json),apply,5/5,3
1,grupo 01 (decision_g1.json),apply,5/5,2


## 5. Recomendação operacional

**Gatilho.** Quando o ISC ultrapassa 1,0 em pelo menos 3 das 5 amostras mais recentes e
as features em concordância incluem `RRU.PrbTotUl`, o sinal é de **saturação de uplink
sustentada** — não de um pico transitório.

**Ação candidata.** Emitir uma política A1 de priorização para o escopo QoS afetado
(`policytype_id` 1, `priorityLevel` 10), sujeita a **validação humana** antes de
qualquer aplicação real.

**Por que a exigência de duas features importa aqui.** A cauda do `iperf` em recovery
dispara uma amostra isolada. Com a regra de concordância e a janela de voto, ela não
aciona a política — que é exatamente o comportamento desejado de um laço de controle:
não reagir a transiente.

**O que esta recomendação não autoriza.** Não afirmamos efeito físico na RAN. O lab não
comprova O1/NETCONF no softmodem OAI monolítico, nem quota de PRB via E2SM-RC
action 6. A política é um **artefato candidato**, não uma atuação.

In [7]:
print("ARTEFATOS GERADOS PELO PACOTE\n" + "-" * 46)
for p in sorted(DERIVED.glob("*")) + sorted(FIGURES.glob("*.png")):
    print(f"  {p.relative_to(RAIZ)!s:<42} {p.stat().st_size:>8,} bytes")
print("\nmodo de atuacao:", decisao_g1["actuation"]["mode"].upper(),
      "— nenhuma requisicao enviada ao Near-RT RIC.")

ARTEFATOS GERADOS PELO PACOTE
----------------------------------------------
  derived\decision_g1.json                      1,770 bytes
  derived\etl_qc.json                             860 bytes
  derived\kpi_por_fase.csv                        194 bytes
  derived\kpm_features.csv                      8,526 bytes
  figures\p1_isc_serie.png                     59,463 bytes
  figures\p2_taa_por_fase.png                  70,870 bytes
  figures\p3_distribuicoes.png                 61,305 bytes
  figures\p4_matriz_features.png               35,630 bytes

modo de atuacao: DRY-RUN — nenhuma requisicao enviada ao Near-RT RIC.
